# HRG Grouper Analysis Run Book

This document serves as a run book for the HRG Grouper Analysis project. It Completes the steps to set up the environment, run the analysis, and interpret the results. This can either be done with real data by an NHS organization, or with the test data provided by the NHS England's Casemix Office. Out of the box this uses the test data.

## Install requirements

In [29]:
import Utils.install_requirements

Utils.install_requirements.install_requirements()


## Download and Extract the Test Data

In [32]:
import requests
import os
import zipfile
import shutil
from Utils.constants import DATA_FILE_FOLDER, SAMPLE_DATA_FILE, RAW_FILE_FOLDER

# URL of the zip file
url = 'https://digital.nhs.uk/binaries/content/assets/website-assets/services/national-casemix-office/hrg4-2024-25-local-payment-grouper/hrg4-202425-local-payment-grouper-test-data-and-expected-results-v1.0.zip'
zip_path = os.path.join(DATA_FILE_FOLDER, 'test_data.zip')
target_file_path_and_name = 'HRG4+ 202425 Local Payment Grouper Test Data and Expected Results v1.0/APC/HRG4+ 202425 Local Payment Grouper Admitted Patient Care Sample Test Data.csv'

# Download the file
response = requests.get(url, stream=True)
response.raise_for_status()
with open(zip_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

with zipfile.ZipFile(zip_path, 'r') as f:
    f.extract(target_file_path_and_name, path=DATA_FILE_FOLDER)


shutil.move(
    os.path.join(DATA_FILE_FOLDER, target_file_path_and_name),
    os.path.join(RAW_FILE_FOLDER, SAMPLE_DATA_FILE)
)

os.remove(zip_path)

directory_to_remove = os.path.join(DATA_FILE_FOLDER, target_file_path_and_name.split('/')[0])
shutil.rmtree(directory_to_remove)

# Print the path where the data has been downloaded
print(f'Test data downloaded to: {RAW_FILE_FOLDER}/{SAMPLE_DATA_FILE}')


Test data downloaded to: ./data/raw/APC_Sample_Test_Data.csv


## Download the HRG Grouper application

In [ ]:
import requests
import os
import zipfile
import shutil
import tkinter as tk
from tkinter import messagebox
from Utils.constants import DATA_FILE_FOLDER, SAMPLE_DATA_FILE

def init_tk_root():
    '''
        Initializes a hidden Tkinter root window.
    '''
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.withdraw()
    return root

# URL of the zip file
url = 'https://digital.nhs.uk/binaries/content/assets/website-assets/services/national-casemix-office/hrg4-2024-25-local-payment-grouper/hrg4-202425-local-payment-grouper.zip'
zip_file = 'hrg_grouper.zip'
target_file_path_and_name = 'HRG4+ 202425 Local Payment Grouper/HRGGrouperSetup.exe'
installer_name = target_file_path_and_name.split('/')[1]

# Download the file
response = requests.get(url, stream=True)
response.raise_for_status()
with open(zip_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

# unzip the file
with zipfile.ZipFile(zip_file, 'r') as f:
    f.extract(target_file_path_and_name)

shutil.move(target_file_path_and_name, installer_name)

# Remove the zip file after extraction
os.remove(zip_file)
directory_to_remove = target_file_path_and_name.split('/')[0]
shutil.rmtree(directory_to_remove)

# Print the path where the installer has been downloaded
print(f'HRG Application Installer: ./{installer_name}')

root = init_tk_root()
run_installer = messagebox.askyesno(
    "Run Installer",
    f"Would you like to run the HRG Grouper installer now?\n{installer_name}",
    parent=root
)

if run_installer:
    # Use os.startfile with 'runas' to request elevation
    try:
        os.startfile(installer_name, 'runas')
    except Exception as e:
        messagebox.showerror("Error", f"Failed to run the installer: \n{e}", parent=root)

root.destroy()


HRG Application Installer: ./HRGGrouperSetup.exe
Run the installer to set up the HRG Grouper application.


## Find and add the HRG Grouper executable to the PATH

In [ ]:
import os
import tkinter as tk
from tkinter import messagebox, filedialog

# Default install location
default_path = r'C:/Program Files/NHS England/HRG4+ 2024_25 Payment Grouper/HRGGrouperc.exe'


def set_grouper_exe_env(path):
    '''
        Sets the GROUPER_EXE environment variable and updates the .env file.
    '''

    os.environ['GROUPER_EXE'] = path
    lines = []
    # Check that it isn't already set in .env
    if os.path.exists('.env'):
        with open('.env', 'r') as f:
            lines = f.readlines()
    with open('.env', 'w') as f:
        found = False
        for line in lines:
            if line.startswith('GROUPER_EXE='):
                f.write(f'GROUPER_EXE="{path}"\n')
                found = True
            else:
                f.write(line)
        if not found:
            f.write(f'GROUPER_EXE="{path}"\n')


def get_grouper_exe_from_env():
    '''
        Reads the GROUPER_EXE environment variable from the .env file.
        Returns the path if found.
    '''
    if not os.path.exists('.env'):
        return ''
    with open('.env', 'r') as f:
        for line in f:
            if line.startswith('GROUPER_EXE='):
                # get the path after the '=' and strip quotes
                return line.split('=', 1)[1].strip().strip('"').strip("'")
    return ''

def init_tk_root():
    '''
        Initializes a hidden Tkinter root window.
    '''
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.withdraw()
    return root

def prompt_for_grouper_exe():
    '''
        Prompts the user to select the HRG Grouper executable file.
        If the user selects a file, it sets the GROUPER_EXE environment variable.
    '''
    root = init_tk_root()
    while True:
        file_path = filedialog.askopenfilename(
            title='Select HRGGrouperc.exe',
            filetypes=[('Executable files', '*.exe')],
            parent=root)
        if not file_path:
            messagebox.showwarning("No file selected", "No file was selected.", parent=root)
            break
        if not file_path.lower().endswith("hrggrouperc.exe"):
            retry = messagebox.askyesno(
                "Confirm file",
                f"The application by default is called 'HRGGrouperc.exe'.\nYou selected: {os.path.basename(file_path)}\n\nIs this file correct?",
                parent=root)
            if not retry:
                continue
        set_grouper_exe_env(file_path)
        break
    root.destroy()

def handle_grouper_not_found(title, message):
    '''
        Prompts the user with a message box when the HRG Grouper application is not found.
    '''
    root = init_tk_root()
    messagebox.showinfo(title, message)
    prompt_for_grouper_exe()
    root.destroy()


grouper_exe_path = get_grouper_exe_from_env()
if grouper_exe_path:
    if not os.path.exists(grouper_exe_path):
        handle_grouper_not_found(
            "Not Found",
            f"HRG Grouper application path in .env does not exist: {grouper_exe_path}\n" \
            f"Please provide the path to HRGGrouperc.exe.")
    else:
        print(f"Found HRG Grouper application path in .env: {grouper_exe_path}")
        set_grouper_exe_env(grouper_exe_path)
elif os.path.exists(default_path):
    print(f"Found HRG Grouper application at default location: {default_path}")
    set_grouper_exe_env(default_path)
else:
    root = init_tk_root()
    answer = messagebox.askquestion(
        "HRG Grouper Application",
        "Have you installed the HRG Grouper application?",
        icon='question', parent=root)
    if answer == "no":
        msg = tk.Toplevel(root)
        msg.title("Download Required")
        tk.Label(msg, text="Please download and install the HRG Grouper application.").pack(padx=20, pady=20)
        msg.after(10000, msg.destroy)
        root.wait_window(msg)
        root.destroy()
    else:
        handle_grouper_not_found(
            "Not Found",
            "We were not able to find the HRG Grouper application at the default location.\n" \
            "Please provide the path to HRGGrouperc.exe.")


### Build tariff key-value store 

In [28]:
from tariff_kv_store import get_tariff_kv_store
try:
    _ = get_tariff_kv_store()
    print("Tariff key-value store initialized successfully.")
except Exception as e:
    print(f"Error initializing tariff key-value store: {e}")
    # If the tariff kv store is not initialized, we can try to initialize it again


Tariff key-value store initialized successfully.


In [25]:
import os
import subprocess

# Example Run Book: HRG Grouper Test Data Processing

# 1. Set up environment and paths

# Define paths
data_file = "test_data/hrg_grouper_test_data.csv"
reverse_engineering_script = "scripts/reverse_engineering.py"
comorbidity_script = "scripts/identify_comorbidities.py"
c_utils_dir = "C_Utils"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

# 2. Run reverse engineering on test data
reverse_engineered_file = os.path.join(output_dir, "reverse_engineered.csv")
subprocess.run([
    "python", reverse_engineering_script,
    "--input", data_file,
    "--output", reverse_engineered_file
], check=True)

# 3. Compile C utilities (if not already compiled)
c_utilities = ["comorbidity_identifier.c"]
for util in c_utilities:
    util_path = os.path.join(c_utils_dir, util)
    exe_path = os.path.splitext(util_path)[0]
    if not os.path.exists(exe_path):
        subprocess.run(["gcc", util_path, "-o", exe_path], check=True)

# 4. Run comorbidity code list identification (C utility)
comorbidity_output = os.path.join(output_dir, "comorbidities.csv")
comorbidity_exe = os.path.join(c_utils_dir, "comorbidity_identifier")
subprocess.run([
    comorbidity_exe,
    reverse_engineered_file,
    comorbidity_output
], check=True)

# 5. (Optional) Further processing or reporting
# subprocess.run([
#     "python", comorbidity_script,
#     "--input", comorbidity_output,
#     "--report", os.path.join(output_dir, "comorbidity_report.txt")
# ], check=True)

print("HRG Grouper test data processing complete.")


CalledProcessError: Command '['python', 'scripts/reverse_engineering.py', '--input', 'test_data/hrg_grouper_test_data.csv', '--output', 'output\\reverse_engineered.csv']' returned non-zero exit status 2.